# NB12 — The Full Pipeline: Project Template
**Machine Learning in Finance · Week 12**  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruiyuan-chen/ml-finance-course/blob/main/notebooks/NB12_project_template.ipynb)

**Finance question:** If this is so easy, why isn't everyone rich?

**Key idea:** Your fund from start to finish, in the order of your pitch: **strategy → data → features → model → ranking → weights → backtest → honest checks → risks**. You choose; the course tools do the coding. Every section below matches a part of the final presentation.

**How to use it:** make a copy (File → Save a copy in Drive), fill in the 🟡 cells, then *Runtime → Run all*. It must run from top to bottom with no errors before you submit.

## 0. Your fund in one sentence
Double-click and fill in.

* **Fund name:** *...*
* **Team:** *...*
* **Strategy in one sentence:** *We buy ... because ...*
* **Why should this work? (the finance idea, not "because ML")** *...*

In [ ]:
# ▶ RUN ME FIRST — loads the course tools and data (about 30 seconds the first time)
COURSE_URL = "https://raw.githubusercontent.com/ruiyuan-chen/ml-finance-course/main/"
import os, sys, urllib.request
here = [f for f in [".", "..", "../.."] if os.path.exists(os.path.join(f, "mlfin.py"))]
if here:
    sys.path.insert(0, here[0])
else:
    urllib.request.urlretrieve(COURSE_URL + "mlfin.py", "mlfin.py")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlfin
mlfin.setup(COURSE_URL)

## 1. Data
S&P 500 stocks, monthly, 2010–2025: the largest 500 companies each month, with features known at the time.

In [ ]:
panel = mlfin.make_features()
print(f"{panel['ticker'].nunique():,} companies, {panel['date'].nunique()} months, {len(panel):,} stock-months")

## 2. Features
Pick the features your strategy needs. Every one is computed only from information available before
the month it predicts.

In [ ]:
# 🟡 YOUR TURN — your feature list (see the table below for what each one means)
FEATURES = ["mom_12_1", "vol_12", "size", "btm", "ep", "growth", "leverage", "beta_12"]

pd.Series(mlfin.FEATURE_DESCRIPTIONS).reindex(FEATURES).to_frame("what it measures")

## 3. Model
Every model on the menu predicts the chance a stock beats SPY next month.

In [ ]:
# 🟡 YOUR TURN
MODEL = "boosting"   # ← "boosting", "forest", "logistic" or "tree"

print(mlfin.MODEL_MENU[MODEL])
model = mlfin.make_model(MODEL)

## 4. From ranking to portfolio
Each rebalance date the model ranks every stock; you buy the top `N_LONG` and weight them.

In [ ]:
# 🟡 YOUR TURN
N_LONG = 30             # how many stocks
WEIGHTING = "equal"     # "equal", "rank", "inverse_vol" or "cluster"
REBALANCE = "Q"         # "M", "Q" or "A"
COST_BPS = 20           # trading cost per dollar traded (keep 20 unless you can justify another number)
MAX_WEIGHT = None       # e.g. 0.05 = no stock above 5%
MAX_PER_CLUSTER = None  # e.g. 4 = at most 4 stocks from one cluster

## 5. The backtest
Walk-forward, 2015–2025: re-trained every January on the past only, trading costs included.

In [ ]:
result = mlfin.walk_forward_backtest(panel, model, features=FEATURES, n_long=N_LONG, weighting=WEIGHTING,
                                     rebalance=REBALANCE, cost_bps=COST_BPS, max_weight=MAX_WEIGHT,
                                     max_per_cluster=MAX_PER_CLUSTER, verbose=False)
mlfin.plot_cumulative(result, title="Our fund vs SPY, after costs")
mlfin.plot_drawdown(result)
result["stats"]

## 6. What the model relies on
Permutation importance (Week 6) for the final year's model.

In [ ]:
mlfin.plot_importance(result, title="What our model relies on")

## 7. Honest checks
Three questions the investment committee will ask. Answer each one in your memo.

In [ ]:
print("1. What if trading costs more?")
display(mlfin.cost_experiment(result, costs=[0, 10, 20, 50], plot=False))
print("2. Could the edge over SPY be luck?")
display(mlfin.excess_return_tstat(result))
print("3. Could a monkey have done it?")
monkey, monkey_funds = mlfin.monkey_portfolios(panel, n_funds=200, n_long=N_LONG, rebalance=REBALANCE,
                                               cost_bps=COST_BPS, result=result)
monkey

## 8. Where the money sits
How many clusters of stocks that move together (Week 9) do your latest picks cover?

In [ ]:
mlfin.portfolio_concentration(result)

## 9. Your search log
Every version you tried counts, not just the one you show. List them here (it is part of the
"honesty" row of the rubric). You can run several models at once below.

| # | What we changed | Annual return | Sharpe | Kept? |
|---|---|---|---|---|
| 1 | *defaults* | *...* | *...* | *...* |
| 2 | *...* | *...* | *...* | *...* |

In [ ]:
# 🟡 OPTIONAL — compare models with your other settings (each extra model adds 5–20 seconds)
TRY = ["boosting", "logistic"]   # ← add "forest" or "tree"

lineup = {}
for name in TRY:
    r = mlfin.walk_forward_backtest(panel, mlfin.make_model(name), features=FEATURES, n_long=N_LONG,
                                    weighting=WEIGHTING, rebalance=REBALANCE, cost_bps=COST_BPS,
                                    max_weight=MAX_WEIGHT, max_per_cluster=MAX_PER_CLUSTER, verbose=False)
    lineup[name] = r["stats"]["Strategy (after costs)"]
pd.DataFrame(lineup)

## 10. The numbers for your pitch

In [ ]:
mlfin.fund_summary(result, panel, monkey, name="Our fund")

## 11. Risks checklist
Tick each box (replace `[ ]` with `[x]`) and add one sentence of evidence.

- [ ] **Time split:** the model never trained on the months it traded (walk-forward). *...*
- [ ] **Leakage:** every feature was known before the prediction date. *...*
- [ ] **Costs:** results are after trading costs; we know what happens at 50 bps. *...*
- [ ] **Turnover:** we know how often we trade and what it costs us. *...*
- [ ] **Drawdown:** we can say how much an investor could have lost, and when. *...*
- [ ] **Luck:** we report the t-statistic and the monkey test, and how many versions we tried. *...*
- [ ] **Concentration:** we know how many clusters our picks cover. *...*
- [ ] **Capacity:** our stocks are large enough to trade millions of dollars without moving prices. *...*
- [ ] **The story:** we can explain in plain English why the strategy should work in the future. *...*